# Boosted Decision Tree

GBDT costruisce alberi in modo sequenziale, correggendo progressivamente gli errori di classificazione del modello costruito fino a quel punto, producendo un modello complesso e molto accurato

In [13]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')
# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Treining


In [14]:
def training(file_path, csv_name):
    # Leggo i csv
    df = pd.read_csv(file_path)

    # Filtro solo le pazienti con PR valido
    df_PRvalido = df[df['PR [SII]'].notna()].copy()

    # Vado a separare le features e target
    features = df_PRvalido.drop(columns=['Patient ID', 'lesion idx', 'tumor/benign', 
                             'GRADE', 'ER [SII]', 'PR [SII]', 'HER2 [SII]', 
                             'isTN', 'KI67 [%]', 'Breast'])

    # Prendo solo PR [SII], convertita in valori interi.
    target = df_PRvalido['PR [SII]'].astype(int)

    # Normalizzo i dati per avere feature confrontabili tra loro 
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # Istanzio il modello Gradient Boosting Decision Tree
    gboost = GradientBoostingClassifier(
        n_estimators=100,           # Numero di alberi
        learning_rate=0.1,          # Tasso di apprendimento
        max_depth=3,                # Profondità massima degli alberi
        min_samples_split=5,        # Minimo campioni per split
        min_samples_leaf=2,         # Minimo campioni per foglia
        subsample=0.8,              # Frazione campioni per albero
        random_state=42
    )

    # Alleno il modello sui dati normalizzati
    gboost.fit(features_scaled, target)

    # Creo la cross-validation a 5 fold stratificata
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(gboost, features_scaled, target, cv=cv, scoring='accuracy')

    # Ritorno risultati sintetici: media, deviazione standard e score per ogni fold
    return {
        'mean_accuracy': scores.mean(),     # Accuratezza media su tutte le fold
        'std_accuracy': scores.std(),       # Variabilità tra le fold
        'scores_per_fold': scores           # Accuratezza tra ciascun fold
    }

# Lettura dei file

In [15]:
# Vado a calcolare tutti i risultati e li salvo nel dizionario
results = {}
print("="*50 + "\nBoosted Decision Tree\n" + "="*50)
for name, file_path in datasets.items():
    results[name] = training(file_path, name)


# Converto il dizionario in un modelo leggibile da pandas
summary_data = []
for name, metrics in results.items():
    summary_data.append({
        "Modello": name,
        "Accuracy Media": metrics['mean_accuracy'],
        "Deviazione Standard (±)": metrics['std_accuracy']
    })
# Converto in un dataframe
df_summary = pd.DataFrame(summary_data).sort_values(by="Deviazione Standard (±)")

# me lo ordino
df_sorted = df_summary.sort_values(by="Deviazione Standard (±)")

# Lo stampo a modo di tabella ordinata
print(df_sorted.to_string(index=False))



# Stampo il csv migliore
if not df_sorted.empty:
    best_model_name = df_sorted.iloc[0]['Modello']
    print(f"\n\nModello più stabile: {best_model_name}")
    print(f"Accuracy media: {results[best_model_name]['mean_accuracy']:.3f} ± {results[best_model_name]['std_accuracy']:.3f}")
    print(f"Scores per fold: {[f'{s:.3f}' for s in results[best_model_name]['scores_per_fold']]}")
 

Boosted Decision Tree
             Modello  Accuracy Media  Deviazione Standard (±)
           t2_medsam        0.358242                 0.056184
     t2_preprocessed        0.374725                 0.056505
         t2_original        0.358242                 0.074320
    original_dynamic        0.402198                 0.130506
preprocessed_dynamic        0.361538                 0.138060
      medsam_dynamic        0.436264                 0.148071


Modello più stabile: t2_medsam
Accuracy media: 0.358 ± 0.056
Scores per fold: ['0.357', '0.357', '0.308', '0.462', '0.308']
